<a href="https://colab.research.google.com/github/ginghilterra-collab/Voxtral-TTS-Colab/blob/main/voxtral_colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Check GPU Runtime

First, make sure you're using a GPU runtime:

In [3]:
# Check if GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA Capability: {torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}")
else:
    print("❌ GPU not available! Please go to Runtime > Change runtime type > GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB
CUDA Capability: 7.5


## Step 2: Install Dependencies

Install vLLM and vLLM-Omni with proper versions:

In [4]:
# Install system dependencies first
!sudo apt-get update && sudo apt-get install -y build-essential

# Install vLLM with T4-compatible settings
!pip install --no-cache-dir vllm==0.18.1

# Install vLLM-Omni from source (latest fixes)
!pip install --no-cache-dir git+https://github.com/vllm-project/vllm-omni.git

# Install additional dependencies
!pip install --no-cache-dir soundfile librosa httpx gradio

print("✅ Dependencies installed successfully!")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

## Step 3: Fix BFloat16 Issue for T4 GPU

The main issue is that T4 GPU doesn't support BFloat16. We need to patch the source code:

In [6]:
import os
import re

# Find the voxtral_tts_audio_tokenizer.py file
import subprocess
result = subprocess.run(['find', '/usr/local/lib/python*', '-name', 'voxtral_tts_audio_tokenizer.py'],
                       capture_output=True, text=True)

if result.returncode == 0 and result.stdout.strip():
    tokenizer_file = result.stdout.strip().split('\n')[0]
    print(f"Found tokenizer file: {tokenizer_file}")

    # Read the file
    with open(tokenizer_file, 'r') as f:
        content = f.read()

    # Fix the BFloat16 issue
    original_line = 'dtype=torch.bfloat16'
    fixed_line = 'dtype=torch.float16  # Fixed for T4 GPU compatibility'

    if original_line in content:
        content = content.replace(original_line, fixed_line)

        # Write back the fixed file
        with open(tokenizer_file, 'w') as f:
            f.write(content)

        print("✅ Fixed BFloat16 issue for T4 GPU")
    else:
        print("⚠️ BFloat16 line not found or already fixed")
else:
    print("❌ Could not find voxtral_tts_audio_tokenizer.py file")

❌ Could not find voxtral_tts_audio_tokenizer.py file


## Step 4: Start the Voxtral Server

Now let's start the server with T4-optimized settings:

In [ ]:
import subprocess
import time
import os

# Start the vLLM server with T4-optimized settings
server_command = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'mistralai/Voxtral-4B-TTS-2603',
    '--omni',
    '--dtype', 'float16',  # Use float16 instead of bfloat16
    '--max-model-len', '2048',  # Reduce context length for memory
    '--enforce-eager',  # Disable compilation for compatibility
    '--config-format', 'mistral',  # Use Mistral config format
    '--load-format', 'mistral',  # Use Mistral load format
    '--gpu-memory-utilization', '0.7',  # Leave some memory for system
    '--host', '0.0.0.0',
    '--port', '8000'
]

print("🚀 Starting Voxtral server...")
print("This may take 5-10 minutes to download and load the model.")
print("Please wait for the server to start.")

# Start server in background
server_process = subprocess.Popen(
    server_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

# Wait for server to start
print("Waiting for server to start...")
time.sleep(60)  # Give it time to start

print("✅ Server startup initiated!")

## Step 5: Test the Server

Let's test if the server is working:

In [ ]:
import requests
import time
import io
import soundfile as sf
from IPython.display import Audio, display

# Wait for server to be ready
max_retries = 30
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:8000/health", timeout=5)
        if response.status_code == 200:
            print("✅ Server is ready!")
            break
    except:
        print(f"Waiting for server... ({i+1}/{max_retries})")
        time.sleep(10)
else:
    print("❌ Server failed to start within expected time")
    # Let's check server logs
    if 'server_process' in locals():
        print("\nServer output:")
        try:
            # Read some output from the server
            import select
            if select.select([server_process.stdout], [], [], 0.1)[0]:
                for line in server_process.stdout:
                    print(line.strip())
                    break
        except:
            pass

## Step 6: Generate Speech

Now let's test the TTS functionality:

In [ ]:
import requests
import io
import soundfile as sf
from IPython.display import Audio, display

# Test text-to-speech
test_text = "Hello! This is Voxtral, Mistral's advanced text-to-speech model. I can speak in multiple languages and different voices."

payload = {
    "input": test_text,
    "model": "mistralai/Voxtral-4B-TTS-2603",
    "response_format": "wav",
    "voice": "casual_male",  # Available voices: casual_male, casual_female, professional_male, etc.
    "speed": 1.0
}

try:
    print("🎵 Generating speech...")
    response = requests.post(
        "http://localhost:8000/v1/audio/speech",
        json=payload,
        timeout=120.0
    )

    if response.status_code == 200:
        # Load and play the audio
        audio_array, sr = sf.read(io.BytesIO(response.content), dtype="float32")
        print(f"✅ Generated audio: {len(audio_array)} samples at {sr} Hz")
        print(f"Duration: {len(audio_array)/sr:.2f} seconds")

        # Display audio player
        display(Audio(data=audio_array, rate=sr))
    else:
        print(f"❌ Error generating speech: {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"❌ Error: {e}")

## Step 7: Try Different Voices and Languages

Let's test different voices and languages:

In [ ]:
# Test different voices and languages
test_cases = [
    {
        "text": "Bonjour! Je suis Voxtral, un modèle de synthèse vocale avancé.",
        "voice": "casual_female",
        "language": "French"
    },
    {
        "text": "¡Hola! Soy Voxtral, un modelo avanzado de texto a voz.",
        "voice": "professional_male",
        "language": "Spanish"
    },
    {
        "text": "Hallo! Ich bin Voxtral, ein fortschrittliches Text-zu-Sprache-Modell.",
        "voice": "casual_male",
        "language": "German"
    }
]

for i, test_case in enumerate(test_cases):
    print(f"\n🎤 Testing {test_case['language']} with {test_case['voice']} voice...")

    payload = {
        "input": test_case["text"],
        "model": "mistralai/Voxtral-4B-TTS-2603",
        "response_format": "wav",
        "voice": test_case["voice"]
    }

    try:
        response = requests.post(
            "http://localhost:8000/v1/audio/speech",
            json=payload,
            timeout=60.0
        )

        if response.status_code == 200:
            audio_array, sr = sf.read(io.BytesIO(response.content), dtype="float32")
            print(f"✅ {test_case['language']} audio generated successfully")
            display(Audio(data=audio_array, rate=sr))
        else:
            print(f"❌ Error with {test_case['language']}: {response.status_code}")

    except Exception as e:
        print(f"❌ Error with {test_case['language']}: {e}")

## Step 8: Voice Cloning (Optional)

Voxtral supports voice cloning with reference audio. Here's how to use it:

In [ ]:
# For voice cloning, you would need to upload a reference audio file
# This is an example of how it would work:

voice_cloning_example = {
    "text": "This is a demonstration of voice cloning capabilities.",
    "voice": "custom",  # Use custom voice
    "# You would need to provide reference_audio_path here"
}

print("💡 Voice Cloning Instructions:")
print("1. Upload a reference audio file (WAV format, 10-30 seconds)")
print("2. Use the reference audio to clone the voice")
print("3. The model will adapt to the new voice characteristics")
print("\nNote: Voice cloning requires additional setup and reference audio files.")

## Troubleshooting

### Common Issues:

1. **RuntimeError: expected scalar type BFloat16 but found Half**
   - Make sure Step 3 (BFloat16 fix) was executed successfully

2. **Out of Memory Error**
   - Restart the runtime and try again
   - Make sure GPU memory utilization is set to 0.7 or lower

3. **Server not responding**
   - Wait longer for the server to start (can take 5-10 minutes)
   - Check the server logs for errors

4. **Missing config files error**
   - This is normal for this model, the `--config-format mistral` and `--load-format mistral` flags handle it

### Performance Tips:
- Use shorter text for faster generation
- The model works best with text under 500 characters
- For longer texts, consider breaking them into smaller chunks

### Available Voices:
- `casual_male`
- `casual_female`
- `professional_male`
- `professional_female`
- And more... (check the model documentation for the full list)

## Cleanup

Stop the server when you're done:

In [ ]:
# Stop the server
if 'server_process' in locals():
    server_process.terminate()
    server_process.wait()
    print("✅ Server stopped successfully")
else:
    print("No server process to stop")

# Clean up GPU memory
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU memory cleared")